Problem Statement

The objective of this project is to develop a deep learning system that can understand and classify textual data based on its sentiment. We will first use the IMDB movie review dataset as a controlled learning problem and progressively build and compare Simple RNN, LSTM, and GRU models using PyTorch.

The project will begin with text preprocessing, tokenization, sequence padding, and word embeddings. We will then train recurrent neural network models to classify movie reviews as positive or negative. The models will be evaluated and compared using appropriate performance metrics, training behavior, and error analysis.

After establishing a strong understanding of sequence modeling, the learned techniques will be applied to a more practical YouTube Comment Analyzer, where comments can be classified according to meaningful categories such as sentiment or toxicity, depending on the available dataset and labeling quality.

IMDB movie reviews = training/learning environment to understand RNNs, LSTMs, and GRUs.

Then we take what we learned and apply it to the real-world YouTube Comment Analyzer.

In [5]:
import random 
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
import torch
print("PyTorch version:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())#torch.cuda.is_available() → checks whether a GPU is available.

PyTorch version: 2.14.0+cpu
CUDA available: False


The GPU check matters because training neural networks can be much faster on a GPU.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
!pip install torch datasets

PyTorch is the deep-learning framework we use to build/train the RNN, while Hugging Face Datasets is a separate library we use to load the IMDB data.

you do not need to install huggingface_hub separately for what we are doing
You already ran:
pip install torch datasets

The datasets package uses huggingface_hub internally, so it is normally installed as a dependency.

In [6]:
import torch
from datasets import load_dataset

print("PyTorch version:", torch.__version__)

r:\C\PYTHON PROGRAMS\Deep Learning Project 4(RNN)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.14.0+cpu


In [7]:
import datasets
import huggingface_hub

print(datasets.__version__)
print(huggingface_hub.__version__)

5.0.1
1.30.0


In [8]:
#Load IMDB
dataset=load_dataset('stanfordnlp/imdb')
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


Yes. stanfordnlp/imdb is the Hugging Face dataset identifier (repository address/name) for the IMDb dataset. It tells load_dataset() exactly which dataset repository to fetch.

In [8]:
print(dataset['train'][0])
print(dataset["train"][1])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

The IMDB dataset contains movie reviews with labels:
text → movie review
label → 0 (negative) or 1 (positive)

In [9]:
print(dataset['test'][24000])

{'text': "The movie is about a girl who's not going to a bonfire only because she's baby-sitting that night. Nothing weird about that, right? Until ... The phone rings. Until ... The phone rings again. And again ... And again. Those are not some stupid prank calls. This is for real. If you wanna see how the girl reacts, just watch the movie.<br /><br />Great atmosphere filled with scary sounds. Very well performed by young Camilla Belle who got the lead role. I see in her some great potential to become a good actress. This is more than only a decent thriller, I have no idea why it's so underrated. Anyway, on my opinion this movie deserves more than only 4/10. 24% of all voters rated the movie with 1. Get serious, people. You couldn't get a better thriller for a title like this.", 'label': 1}


Our project uses:
TRAIN
25,000 labeled reviews
        ↓
Learn

TEST
25,000 labeled reviews
        ↓
Evaluate

We first loaded the raw dataset because we needed to confirm what it contains. Now preprocessing is the next major stage.

Text preprocessing
What are we doing?

Our dataset currently contains:
"I rented I AM CURIOUS-YELLOW from my video store..."
PyTorch cannot send this English sentence directly into an RNN.

We first convert it into tokens:
"I rented this movie"
        ↓
["i", "rented", "this", "movie"]

Then later:
["i", "rented", "this", "movie"]
        ↓
[15, 428, 37, 892]

The numbers are the word IDs in our vocabulary.
Why are we doing this?

Because the RNN ultimately works with numerical tensors, not words.

So our preprocessing pipeline is:

Raw text
   ↓
Cleaning
   ↓
Tokenization
   ↓
Vocabulary
   ↓
Integer encoding
   ↓
Padding
   ↓
Tensor
   ↓
Embedding
   ↓
RNN

One important decision

For this project, I recommend that we build the tokenizer/vocabulary ourselves rather than using a ready-made NLP tokenizer.

In [9]:
text=dataset['train'][0]['text']
print(text)

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [10]:
tex=dataset['train'][0]['label']
print(tex)

0


Now we will perform the first preprocessing operation: basic text cleaning.

Our reviews contain HTML tags such as:
<br /><br />
These are not meaningful words for sentiment classification, so we should remove them.

In [11]:
import re
text=dataset['train'][0]['text']
clean_text=re.sub(r"<br\s*/?>",' ',text)
print(clean_text)

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.  The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.  What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it's not shot

Tokenization
We want to convert:
"I rented I AM CURIOUS-YELLOW from my video store"
into individual tokens:
["i", "rented", "i", "am", "curious-yellow", "from", "my", "video", "store"]

For our first implementation, we'll use a simple Python tokenizer so you understand what is happening rather than hiding it inside a library.

In [12]:
def tokenize(text):
    return text.lower().split()

tokens=tokenize(clean_text)
print(tokens[:20])

['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was']


Why .lower()?
These:
Movie
movie
MOVIE
should normally be treated as the same word.
So we convert everything to lowercase.

Why .split()?
.split() separates the text wherever there is whitespace:

"I love this movie"
        ↓
["i", "love", "this", "movie"]

This is our first simple tokenizer.

Later, when we build the vocabulary, we'll convert these tokens into integer IDs.

So the pipeline currently is:
Raw review
    ↓
Remove HTML
    ↓
Lowercase
    ↓
Split into tokens
    ↓
["i", "rented", "i", "am", ...]

We need:

25,000 raw reviews
        ↓
tokenize() on every review
        ↓
25,000 lists of tokens

In [13]:
# So first let's apply our tokenizer to the whole training set.
train_tokens=[tokenize(text) for text in dataset['train']['text']]
print("Number of reviews:",len(train_tokens))
print("First review tokens:",train_tokens[0][:20])

Number of reviews: 25000
First review tokens: ['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was']


train_tokens[0][:20] means:
train_tokens[0] → first review's complete token list
[:20] → take only the first 20 tokens

Now we will take all tokens from those 25,000 reviews, count how often each word appears, and then create our word-to-ID mapping.

We will do this in two stages:
1. Count word frequencies
2. Keep the most frequent words and assign IDs

For our project, we'll use a fixed vocabulary size such as 10,000 words so the model remains manageable.This means -- we won’t keep every unique word in the dataset.
Suppose there are 50,000 different words across the reviews. We choose only the 10,000 most frequently occurring words as our vocabulary.

Top 10,000 words → get their own IDs
Remaining rare words → <UNK>
This keeps the vocabulary smaller, so the model is faster and uses less memory.

You’re right that train_tokens contains 25,000 separate lists
every individual token inside every review is counted

In [14]:
#we’ll build the vocabulary from the tokenized training reviews only
#This is the step where every word gets its consistent integer ID.

#We are only counting right now. The actual word-to-number mapping comes immediately after this.
from collections import Counter
word_counts=Counter()
for tokens in train_tokens:
    word_counts.update(tokens) #update(tokens) = go through the tokens and add 1 to each word's frequency

print("Unique words:",len(word_counts))
print("Top 10 words:", word_counts.most_common(10))#gives the 10 most frequently occurring words/tokens, along with their counts.

#Counter() itself stores the words and their frequencies, so we don't need to create a separate dictionary.


#Your output
#Unique words: 251637
#means that across the 25,000 training reviews, there are 251,637 different tokens/words in our vocabulary before limiting it to the top 10,000.

#the, appeared 322,198 times across the entire 25,000-review training set.

Unique words: 251637
Top 10 words: [('the', 322198), ('a', 159953), ('and', 158572), ('of', 144462), ('to', 133967), ('is', 104171), ('in', 90527), ('i', 70480), ('this', 69714), ('that', 66292)]


Now comes the important part: Vocabulary
We cannot assign random IDs separately for every review. We need one common vocabulary for the entire training dataset.
For example:
<padded> → 0
<unk>    → 1
the      → 2
movie    → 3
good     → 4
bad      → 5
...
Then every review uses the same mapping.

Vocabulary = one common dictionary that assigns a unique number (ID) to each word.
So if: "the" → 2
then every occurrence of "the" in every review will be represented by 2.


What are <PAD> and <UNK>?
<PAD> = padding token. We use it to fill shorter reviews so all sequences have the same length.

<UNK> = unknown token. Used when a word is not present in our vocabulary (for example, a rare word that we decided not to include).

The important point is:<UNK> does not mean “gibberish.” It means “a valid word that our vocabulary does not contain.”

For example, suppose we keep only the 10,000 most frequent words:
the       → 2
movie     → 3
good      → 4

If "fantabulous" exists in English but is too rare and therefore was not included in our 10,000-word vocabulary, then:
fantabulous → <UNK> → 1
So <UNK> means unknown to our vocabulary, not unknown to the English language.

--Build the actual vocabulary--
Now we take those 251,637 unique words and keep only the 10,000 most frequent.

Then we assign IDs:
<PAD> → 0
<UNK> → 1
most frequent word → 2
next word → 3
...
10,000th word → ID

This is the step that turns our frequency counter into an actual word → number dictionary that we'll use to encode the reviews.

We will create a mapping like:

<PAD> → 0
<UNK> → 1
most frequent word → 2
2nd most frequent → 3
...

In [15]:
MAX_VOCAB_SIZE=10000

# Get the 10,000 most frequent words
most_common_words=word_counts.most_common(MAX_VOCAB_SIZE-2) #most_common_words contains words in frequency order.

# Create word → ID mapping

word_to_id={
    "<PAD>":0,
    "<UNK>":1
}

for idx,(word,count) in enumerate(most_common_words,start=2):
    word_to_id[word]=idx

print("Vocabulary Size:",len(word_to_id))
print("First 10 entries:",list(word_to_id.items())[:10])


Vocabulary Size: 10000
First 10 entries: [('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('a', 3), ('and', 4), ('of', 5), ('to', 6), ('is', 7), ('in', 8), ('i', 9)]


MAX_VOCAB_SIZE - 2 is because we already reserved 2 IDs:
0 → <PAD>
1 → <UNK>
So out of 10,000 total IDs, the actual words get 9,998 IDs.

start=2
means the first actual word gets ID 2 because:
<PAD> → 0
<UNK> → 1

word_to_id[word] = idx
inserts the word as the key and its ID as the value into the dictionary.

.items() → gets (word, ID) pairs

count is the word's frequency, but we're not using it here.you technically don't need count here because we aren't using the frequency anymore.

We need (word, count) only because most_common_words contains pairs like:
("the", 322198)
("a", 159953)

We can write _ instead of count to show we're intentionally ignoring it:
for idx, (word, _) in enumerate(most_common_words, start=2):

Here _ simply means: “There is a value here, but I don't need it.”

#numerical encoding. 
Right now each review is:['i', 'rented', 'i', 'am', 'curious-yellow', ...]

But the RNN needs numbers:
['i', 'rented', 'i', 'am']
        ↓
[9, 742, 9, 317]
We will create a function that looks up each word in word_to_id. If the word isn't there, it uses the <UNK> ID 1.

In [16]:
def encode_review(tokens):
    return [word_to_id.get(word,word_to_id['<UNK>']) for word in tokens]

encoded_review=encode_review(train_tokens[0])
print('First 20 tokens:')
print(train_tokens[0][:20])

print("\nEncoded:")
print(encoded_review[:20])

First 20 tokens:
['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was']

Encoded:
[9, 1486, 9, 226, 1, 34, 57, 433, 1495, 77, 5, 35, 2, 9840, 11, 3471, 12, 50, 12, 14]


.get() is a dictionary method used to safely get the value for a key.

word_to_id.get(word, word_to_id["<UNK>"])
means:Look for word in word_to_id. If it exists, return its ID; otherwise return the ID of <UNK> (which is 1).

SO, word_to_id["<UNK>"]
is not automatically used. It is only used when the word is missing.

In [17]:
#Encode the entire training set, Now we need to do the same for all 25,000 training reviews.

encoded_train=[encode_review(tokens) for tokens in train_tokens]

print("Number of encoded reviews:",len(encoded_train))
print("First encoded review:",encoded_train[0][:20])
print('First 3 review encode:',[review[:20] for review in encoded_train[0:3]])




Number of encoded reviews: 25000
First encoded review: [9, 1486, 9, 226, 1, 34, 57, 433, 1495, 77, 5, 35, 2, 9840, 11, 3471, 12, 50, 12, 14]
First 3 review encode: [[9, 1486, 9, 226, 1, 34, 57, 433, 1495, 77, 5, 35, 2, 9840, 11, 3471, 12, 50, 12, 14], [1179, 226, 1, 1, 7, 3, 1, 4, 2352, 9842, 1, 12, 141, 662, 48, 1942, 954, 3187, 22, 77], [46, 58, 6, 909, 242, 10, 585, 5, 24, 8, 2, 3263, 10, 24, 7, 246, 15, 32, 3837, 18]]


So encoded_train becomes:
[
  [9, 1486, 9, 226, 1, ...],   # review 1
  [2, 45, 78, 91, ...],        # review 2
  [15, 8, 34, ...],             # review 3
  ...
]

Notice that we still have different sequence lengths. That's the next problem we need to solve.
the next step will be:
❗ different lengths → same length
which is where padding comes in.

Our encoded reviews still have different lengths:
Review 1 → 218 IDs
Review 2 → 97 IDs
Review 3 → 341 IDs

But when we create batches for PyTorch, we need a common sequence length.
So we'll choose:
MAX_LEN = 200

Then:
< 200 → add 0 (<PAD>) at the beginning
> 200 → keep only the first 200 IDs
= 200 → unchanged

After padding:
Review 1 → 200 IDs
Review 2 → 200 IDs
Review 3 → 200 IDs
Then we can finally convert them into PyTorch tensors.

Why 200?
It's a design choice: large enough to retain substantial review context, but not so large that computation and memory become unnecessarily expensive.

In [18]:
MAX_LEN=200
def pad_sequence(sequence,max_len=MAX_LEN):
    if len(sequence) < max_len:
        return sequence + [word_to_id["<PAD>"]] * (max_len-len(sequence))
    else:
        return sequence[:max_len]

padded_train=[pad_sequence(sequence) for sequence in encoded_train]
print("Number of reviews:",len(padded_train))
print("Length of first review:", len(padded_train[0]))
print("First 20 IDs:", padded_train[0][:20])



Number of reviews: 25000
Length of first review: 200
First 20 IDs: [9, 1486, 9, 226, 1, 34, 57, 433, 1495, 77, 5, 35, 2, 9840, 11, 3471, 12, 50, 12, 14]


word_to_id["<PAD>"] → 0
max_len - len(sequence) → how many 0s we need to add
* → repeats 0 that many times
+ → appends those 0s to the original sequence

Example:
sequence = [9, 15, 23]
max_len = 5
Then: 5 - 3 = 2

so: [9, 15, 23] + [0, 0]
  → [9, 15, 23, 0, 0]
So yes: we add 0s until the sequence reaches max_len.

Next step — Do the same preprocessing for the test set
This is important: we must use the exact same vocabulary we built from training data. We do not create a new vocabulary for the test data.

In [19]:
test_tokens = [tokenize(text) for text in dataset["test"]["text"]]

encoded_test = [encode_review(tokens) for tokens in test_tokens]

padded_test = [pad_sequence(sequence) for sequence in encoded_test]

print("Number of test reviews:", len(padded_test))
print("Length of first test review:", len(padded_test[0]))

Number of test reviews: 25000
Length of first test review: 200


eventually we'll have:
Training:
padded_train + train labels

Testing:
padded_test + test labels

Convert the data into PyTorch tensors, Right now:
padded_train → Python list of 25,000 sequences
padded_test  → Python list of 25,000 sequences

We now convert them into PyTorch tensors, because PyTorch models work with tensors.

In [23]:
#First get the labels:
train_labels=dataset['train']['label']
test_labels=dataset['test']['label']


In [24]:
X_train=torch.tensor(padded_train,dtype=torch.long)
y_train=torch.tensor(train_labels,dtype=torch.float32)

X_test=torch.tensor(padded_test,dtype=torch.long)
y_test=torch.tensor(test_labels,dtype=torch.float32)

X_train and X_test contain word IDs such as:
[9, 1486, 9, 226, 1, ...]

dtype=torch.long → tells PyTorch: these are integer IDs (like 9, 1486, 226), used to look up words in the Embedding layer.
dtype=torch.float32 → tells PyTorch: these are decimal/continuous numbers; here, the labels 0 and 1 are stored as floating-point values for the loss calculation.

In [25]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: torch.Size([25000, 200])
y_train shape: torch.Size([25000])
X_test shape: torch.Size([25000, 200])
y_test shape: torch.Size([25000])


This means: 25,000 reviews × 200 tokens

Now we move to the first real PyTorch data-pipeline step: Dataset and DataLoader.
Create PyTorch Dataset

We currently have:X_train → 25,000 × 200 integer IDs ,y_train → 25,000 labels
X_test  → 25,000 × 200 integer IDs
y_test  → 25,000 labels
But we don't want to give all 25,000 reviews to the model at once.

Instead, we want small batches, for example:
25,000 reviews
      ↓
batch of 32
batch of 32
batch of 32
...
That is what Dataset and DataLoader help us organize.

In [26]:
#First, create the Dataset
from torch.utils.data import TensorDataset

train_dataset=TensorDataset(X_train,y_train)
test_dataset=TensorDataset(X_test,y_test)

TensorDataset simply pairs the input and its corresponding label:
X_train[0] ↔ y_train[0]
X_train[1] ↔ y_train[1]
X_train[2] ↔ y_train[2]
...
So one dataset item is basically:
(review sequence, correct label)

In [27]:
print(train_dataset[0])

(tensor([   9, 1486,    9,  226,    1,   34,   57,  433, 1495,   77,    5,   35,
           2, 9840,   11, 3471,   12,   50,   12,   14,   82,  702,    8,    1,
           9,   81,  510,   11,   29,   82,   12,   14,    1,   31, 2443,    1,
          46,   12,  125,  737,    6, 2764,   10, 4218, 1935,   99,    3,  376,
           5,  129, 1127,    1,    9,   61,   62,    6,   67,   10,   16,    1,
          13,   93,  131,    7, 6447,  197,    3,  185, 4380,  659, 1669,  699,
        5601,   36,  438,    6,  781,  292,   53,   64,   43,  506,    8,  928,
          53,  438,    6, 1182,   42,    1,    6,  242,   45,  391,    5,  797,
          19,   48,    2,  952,    1,  199,   43,  721,  954, 1600,  130,   15,
           2, 3472,  408,    4, 1977, 1600,    8,    2, 2351, 7328,    8,  187,
        2170, 9841,    4, 2268,    1,    5,    1,   43,   59, 6083,   19, 9045,
          53,   41,  454,   17,   42,  659,    1,    1,    4, 1114,    1,   13,
        1063, 1068,   87,   43,    9,  

You should see two tensors:
(sequence of 200 IDs, label)

In [28]:
#Then create DataLoaders
from torch.utils.data import DataLoader
BATCH_SIZE=32

train_loader=DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Why shuffle=True for training?
We want the training samples to be presented in a different order each epoch, which helps avoid the model relying on the original ordering of the training data.

Why shuffle=False for testing?
Testing is only evaluation, so there is no benefit to randomly rearranging the test samples.

In [29]:
#Create the DataLoader and inspect one batch
X_batch, y_batch = next(iter(train_loader))

print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)

X batch shape: torch.Size([32, 200])
y batch shape: torch.Size([32])


What does this mean?
32  → 32 reviews in one batch
200 → 200 token IDs per review

For example:
Review 1 → 200 IDs → label 0
Review 2 → 200 IDs → label 1
Review 3 → 200 IDs → label 0
...
This is exactly what will be passed into the model batch by batch, rather than all 25,000 reviews at once.

next(iter(train_loader)) means:
Take the first batch from train_loader.

Breakdown:
iter(train_loader)
→ creates an iterator that can go through batches one by one.

next(...)
→ asks that iterator for the next batch.

So:next(iter(train_loader))
→ gives you the first batch of 32 reviews and their 32 labels.

That’s why we used:
X_batch, y_batch = next(iter(train_loader))

It separates that first batch into:
X_batch → 32 reviews × 200 IDs
y_batch → 32 labels

In [30]:
#Create the Embedding layer
#Let's first define our model settings:
VOCAB_SIZE = 10000
EMBEDDING_DIM = 128
HIDDEN_SIZE = 64


In [31]:
embedding = torch.nn.Embedding(
    num_embeddings=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    padding_idx=0
)

num_embeddings=10000
→ Our vocabulary has 10,000 IDs, from 0 to 9999.

embedding_dim=128
→ Each word ID will be converted into a 128-dimensional vector.

padding_idx=0
→ ID 0 is <PAD>, so PyTorch knows that this is our padding token.

Embedding Layer
Our batch currently looks like:

X_batch shape = [32, 200]
That means:
32 reviews
×
200 word IDs

But these IDs are just numbers such as:
[9, 1486, 9, 226, 1, ...]

We don't want the RNN to treat 1486 as “more meaningful” than 9 just because it is numerically larger.

So we use an Embedding layer.
Conceptually:
word ID
   ↓
Embedding lookup
   ↓
vector

For example:

the → 2 → [0.12, -0.41, 0.72, ...]
movie → 150 → [0.31, 0.08, -0.55, ...]

The embedding vectors are learned during training.

In [32]:
embedded=embedding(X_batch)

print("Input Shape:",X_batch.shape)
print("Embedded shape:",embedded.shape)

Input Shape: torch.Size([32, 200])
Embedded shape: torch.Size([32, 200, 128])
